# Computational Gastronomy — Assignment 1 — Question 1

## Q1(a) — Scrape 10,000 recipes

In [ ]:
import json
import random
import re
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.bbcgoodfood.com"
SITEMAP_INDEX = f"{BASE}/sitemap.xml"

HEADERS = {
    "User-Agent": (
        "CGAS-Assignment-1"
       
    )
}

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
URL_LIST_FILE = DATA_DIR / "recipe_urls.txt"
RAW_JSONL = DATA_DIR / "raw_recipes.jsonl"
FAILED_FILE = DATA_DIR / "failed_urls.txt"

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

In [ ]:
def get(url, timeout=15, retries=3):
   
    last_exc = None
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                time.sleep(2 + attempt * 2)
                continue
            if 400 <= r.status_code < 500:
                raise RuntimeError(f"HTTP {r.status_code}")
            last_exc = RuntimeError(f"HTTP {r.status_code}")
        except requests.RequestException as e:
            last_exc = e
        time.sleep(0.5 * (attempt + 1))
    raise last_exc if last_exc else RuntimeError("unknown error")


def discover_recipe_urls(force=False):
  
    if URL_LIST_FILE.exists() and not force:
        urls = URL_LIST_FILE.read_text().splitlines()
        print(f"Loaded {len(urls)} cached recipe URLs from {URL_LIST_FILE}")
        return urls

    idx = get(SITEMAP_INDEX)
    soup = BeautifulSoup(idx.content, "xml")
    recipe_sitemaps = [
        loc.text for loc in soup.find_all("loc") if re.search(r"-recipe\.xml$", loc.text)
    ]
    print(f"Found {len(recipe_sitemaps)} recipe sub-sitemaps")

    all_urls = []
    for sm_url in recipe_sitemaps:
        try:
            r = get(sm_url)
            sm_soup = BeautifulSoup(r.content, "xml")
            all_urls.extend(loc.text for loc in sm_soup.find_all("loc"))
        except Exception as e:
            print(f"  warn: failed to fetch {sm_url}: {e}")
        time.sleep(0.1)

    all_urls = sorted(set(all_urls))
    URL_LIST_FILE.write_text("\n".join(all_urls))
    print(f"Discovered {len(all_urls)} unique recipe URLs -> {URL_LIST_FILE}")
    return all_urls


recipe_urls = discover_recipe_urls()
recipe_urls[:5]

In [ ]:
def flatten_instructions(instr):
    steps = []
    if isinstance(instr, list):
        for s in instr:
            if isinstance(s, dict):
                if s.get("@type") == "HowToSection":
                    steps.extend(flatten_instructions(s.get("itemListElement", [])))
                else:
                    steps.append(s.get("text", "").strip())
            elif isinstance(s, str):
                steps.append(s.strip())
    elif isinstance(instr, str):
        steps.append(instr.strip())
    return [s for s in steps if s]


def extract_recipe(url, html):
    
    soup = BeautifulSoup(html, "html.parser")
    recipe_json = None
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        try:
            data = json.loads(tag.string)
        except (TypeError, ValueError):
            continue
        for item in (data if isinstance(data, list) else [data]):
            if isinstance(item, dict) and item.get("@type") == "Recipe":
                recipe_json = item
                break
        if recipe_json:
            break
    if recipe_json is None:
        return None

    slug = url.rstrip("/").split("/")[-1]
    nutrition = recipe_json.get("nutrition") or {}
    rating = recipe_json.get("aggregateRating") or {}
    author = recipe_json.get("author")
    if isinstance(author, list):
        author = (author[0].get("name") if author else None)

    return {
        "recipe_slug": slug,
        "url": url,
        "title": recipe_json.get("name"),
        "description": recipe_json.get("description"),
        "ingredients": recipe_json.get("recipeIngredient", []),
        "instructions": flatten_instructions(recipe_json.get("recipeInstructions", [])),
        "prepTime": recipe_json.get("prepTime"),
        "cookTime": recipe_json.get("cookTime"),
        "totalTime": recipe_json.get("totalTime"),
        "recipeYield": recipe_json.get("recipeYield"),
        "recipeCategory": recipe_json.get("recipeCategory"),
        "keywords": recipe_json.get("keywords"),
        "author": author,
        "datePublished": recipe_json.get("datePublished"),
        "ratingValue": rating.get("ratingValue"),
        "reviewCount": rating.get("reviewCount"),
        "calories": nutrition.get("calories"),
        "fatContent": nutrition.get("fatContent"),
        "carbohydrateContent": nutrition.get("carbohydrateContent"),
        "proteinContent": nutrition.get("proteinContent"),
    }



for test_url in recipe_urls[:5]:
    try:
        test_html = get(test_url).text
        result = extract_recipe(test_url, test_html)
        if result:
            break
    except Exception as e:
        print(f"  skip {test_url}: {e}")
result

In [ ]:
_write_lock = threading.Lock()


def scrape_one(url):
    r = get(url)
    return url, extract_recipe(url, r.text)


def load_done_urls():
    done = set()
    if RAW_JSONL.exists():
        with open(RAW_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["url"])
                except Exception:
                    pass
    return done


def run(target_count=10000, max_workers=10, seed=42, url_pool=None):
    urls = (url_pool if url_pool is not None else recipe_urls)[:]
    random.Random(seed).shuffle(urls)

    done = load_done_urls()
    remaining = [u for u in urls if u not in done]
    need = max(0, target_count - len(done))
    print(f"Already have {len(done)} recipes on disk. Need {need} more (of {len(remaining)} candidate URLs).")
    if need == 0:
        return len(done)

    todo = remaining[: need + 200]  # small buffer for parse failures
    got, fails = len(done), []
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(scrape_one, u): u for u in todo}
        for i, fut in enumerate(as_completed(futures), 1):
            u = futures[fut]
            try:
                url, rec = fut.result()
                if rec and rec.get("ingredients"):
                    with _write_lock, open(RAW_JSONL, "a", encoding="utf-8") as f:
                        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    got += 1
                else:
                    fails.append(u)
            except Exception:
                fails.append(u)

            if i % 500 == 0 or got >= target_count:
                print(f"[{i}/{len(todo)}] collected={got} elapsed={time.time()-t0:.0f}s")
            if got >= target_count:
                break

    if fails:
        with open(FAILED_FILE, "a", encoding="utf-8") as f:
            f.write("\n".join(fails) + "\n")
    print(f"DONE. Total recipes on disk: {got}. Failed this run: {len(fails)}.")
    return got


n_collected = run(target_count=10000, max_workers=10)

In [ ]:
records = [json.loads(line) for line in open(RAW_JSONL, encoding="utf-8")]
df_raw = pd.DataFrame(records).drop_duplicates(subset="url").reset_index(drop=True)


df_raw = df_raw.sample(frac=1, random_state=42).reset_index(drop=True)
df_raw.insert(0, "recipe_id", range(1, len(df_raw) + 1))

print("Total recipes scraped:", len(df_raw))
df_raw.head()

In [ ]:
# If more than 10000 recipes are scraped we trim to 10000
if len(df_raw) > 10000:
    df_raw = df_raw.iloc[:10000].copy()

print(f"{len(df_raw)} recipes ready for submission")
print(f"(full nested data already on disk -> {RAW_JSONL})")
df_raw.shape

In [ ]:

df_submission = pd.DataFrame({
    "recipe_name": df_raw["title"],
    "recipe_url": df_raw["url"],
    "ingredient_phrases": df_raw["ingredients"].apply(lambda xs: " | ".join(xs)),
    "instructions": df_raw["instructions"].apply(lambda xs: " | ".join(xs)),
    "servings": df_raw["recipeYield"],
    "preparation_time": df_raw["prepTime"],
})

csv_path = DATA_DIR / "raw_recipes.csv"
df_submission.to_csv(csv_path, index=False)
print(f"Saved {len(df_submission)} recipes -> {csv_path}")
df_submission.head()

## Q1(b) — Fine-tune a NER model on the recipe dataset

In [ ]:
# Step 1: fetch non-duplicate (Recipe ID, ingredient phrase) entries -- the
# raw phrases, one per row, BEFORE any NER is applied. Recipe ID and the
# ingredient phrase live in their own separate columns.
raw_phrase_rows = []
for rid, ingredients in zip(df_raw["recipe_id"], df_raw["ingredients"]):
    for phrase in ingredients:
        raw_phrase_rows.append((rid, phrase))

df_raw_phrases = pd.DataFrame(raw_phrase_rows, columns=["recipe_id", "ingredient_phrase"])
df_raw_phrases = df_raw_phrases.drop_duplicates(subset=["recipe_id", "ingredient_phrase"]).reset_index(drop=True)

raw_phrase_path = DATA_DIR / "recipe_id_ingredient_phrase.csv"
df_raw_phrases.to_csv(raw_phrase_path, index=False)
print(f"Saved {len(df_raw_phrases)} non-duplicate (Recipe ID, ingredient phrase) rows -> {raw_phrase_path}")
df_raw_phrases.head(10)

In [ ]:


import spacy
from spacy.tokens import Span

heuristic_nlp = spacy.load("en_core_web_sm", disable=["ner"])

UNITS = set("""
cup cups tablespoon tablespoons tbsp tbsps teaspoon teaspoons tsp tsps
gram grams g kg kilogram kilograms ml milliliter milliliters millilitre millilitres
l liter liters litre litres oz ounce ounces lb lbs pound pounds
pinch pinches dash dashes clove cloves slice slices can cans package packages
stick sticks bunch bunches sprig sprigs handful handfuls piece pieces
pack packs jar jars tin tins bag bags block blocks knob knobs bottle bottles
large medium small whole ground fresh dried plain
""".split())


NUM_ONLY_RE = re.compile(r"^[\d¼-¾⅐-⅞./\-]+$")
QTY_UNIT_RE = re.compile(r"^\d+(\.\d+)?(g|kg|ml|l|oz|lb|cm|mm)$", re.IGNORECASE)
STOPWORDS = [" plus ", " to serve", " to taste", " for ", " if ", " or "]


def normalise_phrase(phrase):
    
    p = re.sub(r"\([^)]*\)", "", phrase).strip()
    p = p.split(",")[0].strip()
    for sw in STOPWORDS:
        idx = p.lower().find(sw)
        if idx > 0:
            p = p[:idx].strip()
    tokens = p.split()
    while tokens:
        t = tokens[0]
        tl = t.lower().strip("-")
        if tl in UNITS or tl in ("x", "\u00d7") or NUM_ONLY_RE.match(t) or QTY_UNIT_RE.match(t):
            tokens.pop(0)
        else:
            break
    return " ".join(tokens)


def heuristic_ingredient_name(phrase):
    cleaned = normalise_phrase(phrase) or phrase
    doc = heuristic_nlp(cleaned)
    chunks = list(doc.noun_chunks)
    name = max(chunks, key=len).text if chunks else cleaned
    return name.strip()


for phrase in df_raw.iloc[0]["ingredients"][:4]:
    print(f"{phrase!r:55s} -> {heuristic_ingredient_name(phrase)!r}")

In [ ]:
def find_span(phrase, name):
   
    if not name:
        return None
    pattern = re.escape(name).replace(r"\ ", r"\s+")
    return re.search(pattern, phrase, flags=re.IGNORECASE)


weak_examples = []  # (text, start_char, end_char)
for ingredients in df_raw["ingredients"]:
    for phrase in ingredients:
        name = heuristic_ingredient_name(phrase)
        m = find_span(phrase, name)
        if m:
            weak_examples.append((phrase, m.start(), m.end()))

print(f"Weakly-labelled {len(weak_examples)} / "
      f"{sum(len(x) for x in df_raw['ingredients'])} ingredient phrases "
      f"({len(weak_examples) / sum(len(x) for x in df_raw['ingredients']) * 100:.1f}% coverage)")

In [ ]:
import warnings
import random
from spacy.training import Example, offsets_to_biluo_tags

nlp = spacy.load("en_core_web_sm")
ner = nlp.get_pipe("ner")
ner.add_label("INGREDIENT")

# Build Example objects, dropping the small fraction where the character span
# does not land on a clean token boundary (can't be used for NER training).
all_examples = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for text, start, end in weak_examples:
        doc = nlp.make_doc(text)
        tags = offsets_to_biluo_tags(doc, [(start, end, "INGREDIENT")])
        if "-" in tags:
            continue
        all_examples.append(Example.from_dict(doc, {"entities": [(start, end, "INGREDIENT")]}))

print(f"{len(all_examples)} / {len(weak_examples)} examples aligned cleanly to token boundaries")

random.Random(0).shuffle(all_examples)
n_dev = max(1000, int(0.05 * len(all_examples)))
dev_examples = all_examples[:n_dev]
train_examples = all_examples[n_dev:]
print(f"train={len(train_examples)}  dev={len(dev_examples)}")

In [ ]:
# Fine-tune: continue training the pretrained NER component (now extended
# with the INGREDIENT label) on our recipe-derived training examples.
N_EPOCHS = 4

with nlp.select_pipes(enable=["tok2vec", "ner"]):
    optimizer = nlp.resume_training()
    for epoch in range(N_EPOCHS):
        random.Random(epoch).shuffle(train_examples)
        losses = {}
        for batch in spacy.util.minibatch(train_examples, size=128):
            nlp.update(batch, sgd=optimizer, losses=losses, drop=0.2)
        ner_loss = losses.get("ner", 0)
        print(f"epoch {epoch+1}/{N_EPOCHS}: ner_loss={ner_loss:.1f}")

In [ ]:
# Evaluate the fine-tuned model on the held-out dev set: exact-span
# precision / recall / F1 for the INGREDIENT label.
tp = fp = fn = 0
for ex in dev_examples:
    doc = nlp(ex.reference.text)
    pred_spans = {(e.start_char, e.end_char) for e in doc.ents if e.label_ == "INGREDIENT"}
    gold_spans = {(e.start_char, e.end_char) for e in ex.reference.ents if e.label_ == "INGREDIENT"}
    tp += len(pred_spans & gold_spans)
    fp += len(pred_spans - gold_spans)
    fn += len(gold_spans - pred_spans)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
print(f"Held-out dev set ({len(dev_examples)} examples) -- exact-span match:")
print(f"  precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}")

In [ ]:
# Save the fine-tuned pipeline to disk
MODEL_DIR = Path("models/ingredient_ner")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
nlp.to_disk(MODEL_DIR)
print(f"Saved fine-tuned NER model -> {MODEL_DIR}")

In [ ]:
# Run the FINE-TUNED model (not the bootstrap heuristic) over every raw
# ingredient phrase in the dataset to produce the final Q1(b) output.
flat_index, flat_phrases = [], []
for i, ingredients in enumerate(df_raw["ingredients"]):
    for phrase in ingredients:
        flat_index.append(i)
        flat_phrases.append(phrase)

t0 = time.time()
flat_names = []
for doc in nlp.pipe(flat_phrases, batch_size=500):
    # sanity filter: a real ingredient name is never a single character (this
    # catches rare leftover noise, e.g. a "2 x 400g cans ..." multiplier "x"
    # slipping through as its own span)
    ing_ents = [e.text.strip() for e in doc.ents if e.label_ == "INGREDIENT" and len(e.text.strip()) > 1]
    flat_names.append(max(ing_ents, key=len).lower() if ing_ents else "")
print(f"Fine-tuned model extracted names for {len(flat_names)} phrases in {time.time()-t0:.1f}s")
print(f"({sum(1 for n in flat_names if n)} phrases got a predicted INGREDIENT span)")

ingredient_names_per_recipe = [[] for _ in range(len(df_raw))]
for i, name in zip(flat_index, flat_names):
    ingredient_names_per_recipe[i].append(name)
df_raw["ingredient_names"] = ingredient_names_per_recipe

sample = df_raw.iloc[0]
print("\nRecipe:", sample["title"])
for raw, name in zip(sample["ingredients"], sample["ingredient_names"]):
    print(f"  {raw!r:55s} -> {name!r}")

In [ ]:
# Save the NER output (per-recipe list of extracted ingredient names) alongside the raw data
df_ner = df_raw[["recipe_id", "title", "ingredients", "ingredient_names"]].copy()
df_ner_flat = df_ner.copy()
df_ner_flat["ingredients"] = df_ner_flat["ingredients"].apply(lambda xs: " | ".join(xs))
df_ner_flat["ingredient_names"] = df_ner_flat["ingredient_names"].apply(lambda xs: " | ".join(xs))
ner_csv_path = DATA_DIR / "ner_ingredient_names.csv"
df_ner_flat.to_csv(ner_csv_path, index=False)
print(f"Saved NER output -> {ner_csv_path}")
df_ner_flat.head()

## Q1(c) — Store recipes as (Recipe ID) — (Ingredient Name) pairs

In [ ]:
recipe_ingredient_pairs = (
    df_raw[["recipe_id", "ingredient_names"]]
    .explode("ingredient_names")
    .rename(columns={"ingredient_names": "ingredient_name"})
    .dropna(subset=["ingredient_name"])
)
recipe_ingredient_pairs = recipe_ingredient_pairs[recipe_ingredient_pairs["ingredient_name"].str.len() > 0]
recipe_ingredient_pairs = recipe_ingredient_pairs.reset_index(drop=True)

pairs_csv_path = DATA_DIR / "recipe_id_ingredient_name.csv"
recipe_ingredient_pairs.to_csv(pairs_csv_path, index=False)

print(f"Saved {len(recipe_ingredient_pairs)} (Recipe ID, Ingredient Name) rows -> {pairs_csv_path}")
print(f"Covering {df_raw['recipe_id'].nunique()} recipes and {recipe_ingredient_pairs['ingredient_name'].nunique()} unique extracted ingredient names")
recipe_ingredient_pairs.head(10)

In [ ]:
# A random sample of 100 recipes, written out as a plain text file: for each
# recipe, a "Recipe <ID>: <title>" heading and its URL, followed by a
# (Recipe ID) -- (Ingredient Name) table for that recipe.
sample_recipe_ids = df_raw["recipe_id"].sample(n=100, random_state=42).sort_values().tolist()
df_by_id = df_raw.set_index("recipe_id")
pairs_by_id = recipe_ingredient_pairs.groupby("recipe_id")

lines = []
for rid in sample_recipe_ids:
    row = df_by_id.loc[rid]
    lines.append(f"Recipe {rid}: {row.title}")
    lines.append(row.url)
    lines.append("")
    lines.append(f"{'Recipe ID':<12}{'Ingredient Name'}")
    lines.append(f"{'-'*9:<12}{'-'*15}")
    if rid in pairs_by_id.groups:
        for r in pairs_by_id.get_group(rid).itertuples():
            lines.append(f"{r.recipe_id:<12}{r.ingredient_name}")
    lines.append("")
    lines.append("=" * 60)
    lines.append("")

sample_txt_path = DATA_DIR / "recipe_id_ingredient_name_sample100.txt"
sample_txt_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Saved a 100-recipe (Recipe ID, Ingredient Name) sample -> {sample_txt_path}")
print("\n".join(lines[:12]))

Saved a 100-recipe (Recipe ID, Ingredient Name) sample -> data/recipe_id_ingredient_name_sample100.txt
Recipe 36: Cheddar & harissa garlic bread
https://www.bbcgoodfood.com/recipes/cheddar-harissa-garlic-bread

Recipe ID   Ingredient Name
---------   ---------------
36          white bread
36          salted butter
36          harissa
36          garlic cloves grated
36          mature cheddar
36          mozzarella

